<a href="https://colab.research.google.com/github/Cristiano-Corsi-Unipi/MIRCV_RAG_project/blob/main/MIRCV_RAG_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compliance Checker: Automated Decision Tree Verification

**Authors:** Lorenzo Ceccanti, Cristiano Corsi, Rojan Shrestha

*Master of Science in Artificial Intelligence and Data Engineering*

---

## Introduction

This project implements an **automated compliance checking system** that leverages **Retrieval-Augmented Generation (RAG)** combined with decision tree evaluation. The system is designed to verify product documentation against regulatory requirements in an automated and systematic way.

### Overview

The pipeline operates through several key stages:

1. **PDF Document Preprocessing**: Using `Unstructured` library with GPT-4 Vision for handling complex elements like images and tables
2. **RAG System Construction**: Creating a vector-based retrieval system using sentence transformers and ChromaDB
3. **Decision Tree Parsing**: Parsing PlantUML-formatted decision trees that encode compliance requirements
4. **Automated Evaluation**: Using an LLM to evaluate each requirement with retrieved context
5. **Tree Navigation**: Automatically navigating decision trees based on LLM responses (YES/NO/NOT APPLICABLE/INSUFFICIENT INFO)

### Use Cases

This system can be applied to:
- Automated compliance verification for product documentation
- Regulatory requirement checking
- Quality assurance automation
- Document-based decision support systems

In the following sections, we will detail each component of the pipeline, from setup to execution.

---

## 1. Setup and Installation

Before diving into the implementation, we need to set up our environment with all required dependencies. The system relies on several key libraries:

- **OpenAI**: For LLM-based evaluation using Azure OpenAI GPT-4
- **Unstructured**: For PDF document parsing and element extraction
- **Sentence Transformers**: For generating text embeddings
- **ChromaDB**: For vector storage and similarity search
- **PyMuPDF**: For PDF manipulation and image extraction

### Installing Dependencies

Run the following cell to install all required packages:

In [ ]:
# Install required packages
!pip install unstructured unstructured[pdf] sentence-transformers chromadb langchain langchain-openai PyMuPDF tiktoken tqdm pandas numpy python-dotenv

### Importing Libraries

Now we import all the necessary modules. The imports are organized by functionality:
- **Standard library** modules for basic operations
- **Data handling** with pandas and numpy
- **PDF processing** with PyMuPDF and Unstructured
- **Embeddings and vector store** with sentence-transformers and ChromaDB
- **LLM interaction** with Azure OpenAI client

In [ ]:
import os
import json
import base64
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass, field
from tqdm import tqdm

import fitz  # PyMuPDF
import pandas as pd
import numpy as np
from openai import AzureOpenAI

from unstructured.partition.auto import partition
from unstructured.documents.elements import Image, Table, Footer, Header, PageBreak, Formula

from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

---

## 2. UML Parser: Decision Tree Data Structures

A crucial component of our compliance checker is the ability to parse and navigate decision trees encoded in **PlantUML** format. These decision trees represent the logical flow of compliance requirements, where each node is either:

- **A question/requirement** that needs to be evaluated against the documentation
- **A verdict** (PASS, FAIL, NOT APPLICABLE) representing the final compliance status

### Why PlantUML?

PlantUML provides a human-readable format for encoding complex decision logic. Our parser supports:
- `if/else/endif` constructs for binary decisions
- `switch/case/endswitch` constructs for multi-way decisions
- Activity nodes (`:text;`) for intermediate statements
- Verdict nodes with color coding (green=PASS, pink=FAIL)
- `detach;` for explicit branch termination

### Data Structures

We define two main classes:
- **`Edge`**: Represents a connection between nodes with an optional label
- **`UMLNode`**: Represents a node in the decision tree with its type, text content, and outgoing edges

In [ ]:
##############################
# Data structures (tree)
##############################

@dataclass
class Edge:
    """Represents an edge in the decision tree graph.
    
    Attributes:
        label: Optional label for the edge (e.g., "Yes", "No")
        target: The UMLNode this edge points to
    """
    label: Optional[str]
    target: "UMLNode"


@dataclass
class UMLNode:
    """A node in the decision tree.

    Attributes:
        id: Unique identifier for the node
        text: The text content of the node (question, statement, or verdict)
        kind: Type of node - one of:
            - "start": synthetic start node
            - "action": plain activity/statement (":" ... ";")
            - "decision": an if-question node
            - "switch": a multi-branch question node
            - "verdict": terminal verdict like PASS/FAIL/NOT APPLICABLE
            - "terminal": explicit end due to 'detach;'
            - "join": implicit merge after if/switch
        edges: List of outgoing edges to other nodes
    """

    id: int
    text: str
    kind: str
    edges: List[Edge] = field(default_factory=list)

    def add_child(self, child: "UMLNode", label: Optional[str] = None) -> None:
        """Add a child node with an optional edge label."""
        self.edges.append(Edge(label=label, target=child))

    def to_dict(self) -> Dict[str, Any]:
        """Convert node to dictionary representation for serialization."""
        return {
            "id": self.id,
            "text": self.text,
            "kind": self.kind,
            "edges": [{"label": e.label, "target": e.target.id} for e in self.edges],
        }

### The UML Parser

The `UMLParser` class is responsible for parsing PlantUML flowchart syntax and building a navigable tree structure. The parser uses a **stack-based approach** to handle nested control structures (if/else, switch/case).

#### Parsing Strategy

The parser maintains a context stack where each context tracks:
- The type of control structure (`"if"` or `"switch"`)
- The control node (decision or switch node)
- A join node for merging branches after the control structure
- The active label for the current branch

This allows proper handling of nested decision structures commonly found in compliance trees.

In [ ]:
##############################
# Parser
##############################

class UMLSyntaxError(Exception):
    """Custom exception for UML parsing errors."""
    pass


class UMLParser:
    """
    Minimal PlantUML-flowchart parser tailored for decision trees.
    
    This parser validates structure and builds a navigable tree from PlantUML
    decision tree files (like SUM/TCM compliance examples).

    Supported constructs (single line, trimmed, case-sensitive):
      - @startuml ... @enduml (required)
      - Activity nodes:  ":some text;"  (leading colon and trailing semicolon)
      - if (QUESTION) then (LABEL)
      - else (LABEL)
      - endif
      - switch (QUESTION)
      - case (LABEL)
      - endswitch
      - Verdict lines starting with one of:
            #lightgreen: PASS ...
            #pink: FAIL ...
            #application: NOT APPLICABLE ...
      - detach; (explicit branch termination)
    """

    def __init__(self) -> None:
        self._next_id = 1

    def _new(self, text: str, kind: str) -> UMLNode:
        """Create a new node with auto-incremented ID."""
        n = UMLNode(id=self._next_id, text=text, kind=kind)
        self._next_id += 1
        return n

    def parse_file(self, path: str) -> "UMLDecisionTree":
        """Parse a PlantUML file and return a UMLDecisionTree."""
        if not os.path.exists(path):
            raise FileNotFoundError(path)
        with open(path, "r", encoding="utf-8") as f:
            content = f.read()
        return self.parse_string(content, source=path)

    def parse_string(self, content: str, source: str = "<memory>") -> "UMLDecisionTree":
        """Parse PlantUML content string and return a UMLDecisionTree."""
        # First, normalize multi-line statements by replacing literal \n with space
        content = content.replace("\\n", " ")
        
        lines = [ln.strip() for ln in content.splitlines()]
        if not self._is_plantuml(lines):
            raise UMLSyntaxError(
                f"{source}: missing @startuml/@enduml or malformed PlantUML block"
            )
        # trim to inside of markers and drop empty/comment-only lines
        body = self._extract_body(lines)
        
        # Join lines that are continuations (don't start with keywords and previous line doesn't end properly)
        merged_body = []
        i = 0
        while i < len(body):
            line = body[i]
            # Keywords that should never be merged (they are complete statements)
            complete_keywords = ("if ", "else ", "else(", "endif", "switch ", "case ", "endswitch", ":", "#", "detach", "'", "@")
            # Check if next line is a continuation (doesn't start with keywords)
            while i + 1 < len(body):
                next_line = body[i + 1]
                # If next line doesn't start with a keyword and current line doesn't end with ; or ) and current line is not a complete keyword, merge them
                if (next_line and 
                    not next_line.startswith(complete_keywords) and
                    not line.endswith((";", ")")) and
                    not line in ("endif", "endswitch")):  # Don't merge after endif/endswitch
                    line = line + " " + next_line
                    i += 1
                else:
                    break
            merged_body.append(line)
            i += 1
        
        tokens = [
            ln for ln in merged_body if ln and not ln.startswith("'")
        ]  # ignore PlantUML comments starting with '
        return self._parse_tokens(tokens, source)

    def _is_plantuml(self, lines: List[str]) -> bool:
        """Check if the content is valid PlantUML."""
        try:
            i0 = lines.index("@startuml")
            i1 = len(lines) - 1 - list(reversed(lines)).index("@enduml")
        except ValueError:
            return False
        return i0 < i1

    def _extract_body(self, lines: List[str]) -> List[str]:
        """Extract the body content between @startuml and @enduml."""
        i0 = lines.index("@startuml")
        i1 = len(lines) - 1 - list(reversed(lines)).index("@enduml")
        return [ln for ln in lines[i0 + 1 : i1]]

    # Helpers to recognize forms quickly
    def _starts(self, s: str, prefix: str) -> bool:
        return s.startswith(prefix)

    def _is_activity(self, s: str) -> bool:
        return self._starts(s, ":") and s.endswith(";")

    def _is_if(self, s: str) -> bool:
        return self._starts(s, "if ") and " then " in s and s.endswith(")")

    def _split_if(self, s: str) -> Tuple[str, str]:
        """Parse if (QUESTION) then (LABEL) syntax."""
        try:
            pre, post = s.split(" then ", 1)
            q = pre[len("if ") :].strip()
            label = post.strip()
            if not (q.startswith("(") and q.endswith(")")):
                raise ValueError
            if not (label.startswith("(") and label.endswith(")")):
                raise ValueError
            return q[1:-1].strip(), label[1:-1].strip()
        except Exception:
            raise UMLSyntaxError(f"Malformed if/then line: {s}")

    def _is_else(self, s: str) -> bool:
        return self._starts(s, "else ") and s.endswith(")")

    def _split_else(self, s: str) -> str:
        """Parse else (LABEL) syntax."""
        lab = s[len("else ") :].strip()
        if not (lab.startswith("(") and lab.endswith(")")):
            raise UMLSyntaxError(f"Malformed else line: {s}")
        return lab[1:-1].strip()

    def _is_switch(self, s: str) -> bool:
        return self._starts(s, "switch ") and s.endswith(")")

    def _split_switch(self, s: str) -> str:
        """Parse switch (QUESTION) syntax."""
        q = s[len("switch ") :].strip()
        if not (q.startswith("(") and q.endswith(")")):
            raise UMLSyntaxError(f"Malformed switch line: {s}")
        return q[1:-1].strip()

    def _is_case(self, s: str) -> bool:
        return self._starts(s, "case ") and s.endswith(")")

    def _split_case(self, s: str) -> str:
        """Parse case (LABEL) syntax."""
        lab = s[len("case ") :].strip()
        if not (lab.startswith("(") and lab.endswith(")")):
            raise UMLSyntaxError(f"Malformed case line: {s}")
        return lab[1:-1].strip()

    def _is_verdict(self, s: str) -> Optional[Tuple[str, str]]:
        """Check if line is a verdict and return (kind, text) if so."""
        # Check for verdict lines with or without space before colon
        # e.g., "#lightgreen:" or "#application :"
        for prefix_base, kind in (
            ("#lightgreen", "PASS"),
            ("#pink", "FAIL"),
            ("#application", "NOT APPLICABLE"),
        ):
            # Try with and without space before colon
            for prefix in [prefix_base + ":", prefix_base + " :"]:
                if s.startswith(prefix):
                    # Extract text after the colon
                    colon_idx = s.index(":")
                    return kind, s[colon_idx + 1:].strip()
        return None

    def _parse_tokens(self, tokens: List[str], source: str) -> "UMLDecisionTree":
        """Main parsing logic - builds the decision tree from tokens."""
        start = self._new(text="START", kind="start")
        current: UMLNode = start
        ctx_stack: List[Tuple[str, UMLNode, UMLNode, Optional[str]]] = []
        # tuple: (kind, control_node, join_node, active_label)
        #   - active_label: branch label for next child emitted under current control

        def ensure_join(control: UMLNode) -> UMLNode:
            # find existing join belonging to top-of-stack or create a new one
            join = self._new(text=f"JOIN after {control.kind}", kind="join")
            return join

        i = 0
        while i < len(tokens):
            line = tokens[i]
            i += 1

            if line == "":
                continue

            if line == "endif":
                if not ctx_stack or ctx_stack[-1][0] != "if":
                    raise UMLSyntaxError(f"{source}: 'endif' without matching 'if'")
                _, control, _, _ = ctx_stack.pop()
                join = ensure_join(control)
                # Link if control to join if there are branches without explicit termination
                control.add_child(join, label=None)  # unlabeled fall-through
                current = join
                continue

            if line == "endswitch":
                if not ctx_stack or ctx_stack[-1][0] != "switch":
                    raise UMLSyntaxError(
                        f"{source}: 'endswitch' without matching 'switch'"
                    )
                _, control, _, _ = ctx_stack.pop()
                join = ensure_join(control)
                control.add_child(join, label=None)
                current = join
                continue

            # Handle both "detach;" and "detach" (with or without semicolon)
            if line == "detach;" or line == "detach":
                term = self._new(text="DETACH", kind="terminal")
                current.add_child(term, label=None)
                # After detach, set current to the terminal node
                # The endif/endswitch will handle proper cleanup
                current = term
                continue

            if self._is_activity(line):
                text = line[1:-1].strip()  # remove leading ':' and trailing ';'
                node = self._new(text=text, kind="action")
                current.add_child(node)
                current = node
                continue

            if self._is_if(line):
                q, then_label = self._split_if(line)
                decision = self._new(text=q, kind="decision")
                current.add_child(decision)
                join = self._new(text=f"JOIN after IF", kind="join")
                ctx_stack.append(("if", decision, join, then_label))
                # move current to decision; next content belongs to 'then' branch
                current = decision
                # mark active label so that first emitted node becomes that branch
                ctx_stack[-1] = ("if", decision, join, then_label)
                continue

            if self._is_else(line):
                if not ctx_stack or ctx_stack[-1][0] != "if":
                    raise UMLSyntaxError(f"{source}: 'else' without matching 'if'")
                kind, decision, join, _ = ctx_stack[-1]
                # switch current back to decision to attach an alternative branch
                current = decision
                new_label = self._split_else(line)
                ctx_stack[-1] = (kind, decision, join, new_label)
                continue

            if self._is_switch(line):
                q = self._split_switch(line)
                sw = self._new(text=q, kind="switch")
                current.add_child(sw)
                join = self._new(text=f"JOIN after SWITCH", kind="join")
                ctx_stack.append(("switch", sw, join, None))
                current = sw
                continue

            if self._is_case(line):
                if not ctx_stack or ctx_stack[-1][0] != "switch":
                    raise UMLSyntaxError(f"{source}: 'case' without matching 'switch'")
                kind, sw, join, _ = ctx_stack[-1]
                current = sw
                lab = self._split_case(line)
                ctx_stack[-1] = (kind, sw, join, lab)
                continue

            verdict = self._is_verdict(line)
            if verdict:
                vkind, rest = verdict
                node = self._new(text=f"{vkind}: {rest}", kind="verdict")
                # If inside a control with an active branch label, attach under that label
                if ctx_stack and ctx_stack[-1][3] is not None:
                    label = ctx_stack[-1][3]
                    ctx_stack[-1] = (
                        ctx_stack[-1][0],
                        ctx_stack[-1][1],
                        ctx_stack[-1][2],
                        None,
                    )
                    ctx_stack[-1][1].add_child(node, label=label)
                else:
                    current.add_child(node)
                current = node
                continue

            # Default: a raw statement in a branch head -> treat like activity
            if ctx_stack and ctx_stack[-1][3] is not None:
                # First node of the branch: attach with label
                label = ctx_stack[-1][3]
                ctx_stack[-1] = (
                    ctx_stack[-1][0],
                    ctx_stack[-1][1],
                    ctx_stack[-1][2],
                    None,
                )
                node = self._new(text=line, kind="action")
                ctx_stack[-1][1].add_child(node, label=label)
                current = node
                continue

            # If we reach here, the token is unsupported
            raise UMLSyntaxError(f"{source}: unsupported or malformed line: {line}")

        # Unwound stack means unmatched endif/endswitch
        if ctx_stack:
            kinds = ", ".join(k for k, *_ in ctx_stack)
            raise UMLSyntaxError(f"{source}: unmatched blocks left open: {kinds}")

        return UMLDecisionTree(start)

### Decision Tree Wrapper

The `UMLDecisionTree` class wraps the parsed tree structure and provides:
- An index for quick node lookup by ID
- Serialization methods for saving/loading trees
- A traversal method for interactive navigation
- Pretty printing for debugging

In [ ]:
##############################
# Decision Tree wrapper
##############################

class UMLDecisionTree:
    """Wrapper class for navigating and manipulating parsed decision trees."""
    
    def __init__(self, root: UMLNode) -> None:
        self.root = root
        self.start = root  # Add 'start' alias for compatibility
        self._index: Dict[int, UMLNode] = {}
        self._build_index(root)

    def _build_index(self, node: UMLNode) -> None:
        """Recursively build an index of all nodes by ID."""
        if node.id in self._index:
            return
        self._index[node.id] = node
        for e in node.edges:
            self._build_index(e.target)

    def to_dict(self) -> Dict[str, Any]:
        """Convert tree to dictionary representation for serialization."""
        nodes = {}
        for nid, node in self._index.items():
            nodes[nid] = node.to_dict()
        return {"root": self.root.id, "nodes": nodes}

    def pretty_print(self) -> None:
        """Print a readable outline of the tree structure."""
        seen = set()

        def dfs(n: UMLNode, indent: str = "") -> None:
            if n.id in seen:
                print(f"{indent}[{n.kind}] {n.text} (↩)")
                return
            seen.add(n.id)
            print(f"{indent}[{n.kind}] {n.text} (id={n.id})")
            for e in n.edges:
                lab = f" --{e.label}--> " if e.label else " --> "
                print(f"{indent}{lab}")
                dfs(e.target, indent + "    ")

        dfs(self.root)

    def traverse(self, answer_fn) -> UMLNode:
        """
        Traverse interactively using an answer function that picks the next edge.
        
        The function receives the current node and its edges, and must return the
        index of the chosen edge (0-based), or None to stop.
        
        Returns:
            The last visited node.
        """
        node = self.root
        while True:
            if not node.edges:
                return node
            idx = answer_fn(node, node.edges)
            if idx is None or idx < 0 or idx >= len(node.edges):
                return node
            node = node.edges[idx].target

---

## 3. Configuration

In this section, we configure all the parameters needed for our compliance checking system. This includes:

- **Azure OpenAI credentials**: API endpoint, key, and version for accessing GPT-4
- **File paths**: Location of the PDF document and decision trees
- **RAG parameters**: Embedding model, chunk size, and retrieval settings
- **LLM parameters**: Model name and temperature for generation

> **Note**: We use `python-dotenv` to load sensitive credentials from a `.env` file. Make sure to create this file with your Azure OpenAI credentials before running.

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Azure OpenAI API Configuration
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_KEY = os.getenv("AZURE_OPENAI_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

# Initialize the Azure OpenAI client
client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

# File Paths Configuration
PDF_PATH = "./assets/ProductDescription.pdf"  # Path to the PDF document to analyze
DECISION_TREES_DIR = Path("./assets/DecisionTrees")  # Directory containing PlantUML decision trees
OUTPUT_DIR = Path("output")  # Directory for saving results
OUTPUT_DIR.mkdir(exist_ok=True)

# RAG System Configuration
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # Lightweight but effective embedding model
TOP_K_DOCUMENTS = 5  # Number of documents to retrieve for each query
CHUNK_SIZE = 1000  # Characters per chunk for document splitting

# LLM Configuration
LLM_MODEL = "gpt-4.1-mini"  # Azure OpenAI deployment name
LLM_TEMPERATURE = 0.1  # Low temperature for more deterministic responses

---

## 4. PDF Preprocessing with Azure OpenAI Vision

One of the critical challenges in processing product documentation is handling **complex visual elements** such as images, diagrams, and tables. Standard text extraction often fails to capture the semantic content of these elements.

### Our Approach

We use a **hybrid preprocessing pipeline** that combines:
1. **Unstructured**: For initial document partitioning and element extraction
2. **Azure OpenAI GPT-4 Vision**: For understanding and transcribing visual content

### Why Use Vision Models?

- **Images**: May contain important diagrams, flowcharts, or technical illustrations
- **Tables**: Structured data that loses meaning when extracted as plain text
- **Formulas**: Mathematical or technical notation that needs proper interpretation

The following functions encode images to base64 and use GPT-4 Vision to extract meaningful text descriptions.

In [ ]:
def encode_image_base64(image_bytes: bytes) -> str:
    """Encode image bytes to base64 string for API transmission."""
    return base64.b64encode(image_bytes).decode('utf-8')


def extract_image_with_gpt4_vision(image_bytes: bytes) -> str:
    """
    Use Azure OpenAI GPT-4 Vision to extract text and descriptions from an image.
    
    Args:
        image_bytes: Raw image data as bytes
        
    Returns:
        Extracted text description of the image content
    """
    base64_image = encode_image_base64(image_bytes)

    response = client.chat.completions.create(
        model="gpt-4.1-mini",  # Azure deployment name
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Extract all text from this image. If it's a diagram or chart, describe it in detail. If it's a table, convert it to structured text format."
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{base64_image}"
                        }
                    }
                ]
            }
        ],
        max_tokens=1000
    )

    return response.choices[0].message.content


def extract_table_with_gpt4_vision(image_bytes: bytes) -> str:
    """
    Use Azure OpenAI GPT-4 Vision to extract and structure table data.
    
    Args:
        image_bytes: Raw image data of the table
        
    Returns:
        Structured text representation of the table
    """
    base64_image = encode_image_base64(image_bytes)

    response = client.chat.completions.create(
        model="gpt-4.1-mini",  # Azure deployment name
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "This is a table. Extract all data and present it as structured text with clear headers and rows. Preserve all information and relationships."
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{base64_image}"
                        }
                    }
                ]
            }
        ],
        max_tokens=1500
    )

    return response.choices[0].message.content

### Main PDF Preprocessing Function

The `preprocess_pdf_with_vision` function orchestrates the entire preprocessing pipeline:

1. Opens the PDF using PyMuPDF
2. Partitions each page using Unstructured with the "fast" strategy
3. Processes each extracted element based on its type:
   - **Images**: Sends to GPT-4 Vision for description
   - **Tables**: Renders as image and extracts structured content
   - **Text elements**: Keeps as-is (excluding headers, footers, formulas)
4. Returns a list of processed elements with metadata

In [ ]:
def preprocess_pdf_with_vision(pdf_path: str) -> List[Dict[str, Any]]:
    """
    Preprocess PDF using Unstructured and Azure OpenAI Vision for images/tables.
    
    This function:
    1. Partitions the PDF into elements using Unstructured
    2. Filters out unwanted elements (footers, headers, formulas)
    3. Uses GPT-4 Vision to extract content from images and tables
    4. Collects text from regular text elements
    
    Args:
        pdf_path: Path to the PDF file
        
    Returns:
        List of processed elements with type, content, and page metadata
    """
    print(f"Processing PDF: {pdf_path}")

    # Open PDF for image extraction
    doc = fitz.open(pdf_path)

    # Partition with Unstructured
    elements = []
    total_pages = len(doc)

    for page_number in tqdm(range(total_pages), desc="Processing pages"):
        # Extract single page
        temp_doc = fitz.open()
        temp_doc.insert_pdf(doc, from_page=page_number, to_page=page_number)

        temp_page_file = f"temp_page_{page_number}.pdf"
        temp_doc.save(temp_page_file)
        temp_doc.close()

        # Partition the page - using 'fast' strategy to avoid Tesseract/OCR requirements
        page_elements = partition(
            filename=temp_page_file,
            strategy="fast",
            include_page_breaks=True
        )

        elements.extend(page_elements)
        os.remove(temp_page_file)

    doc.close()

    # Process elements and use Azure OpenAI Vision for images and tables
    processed_elements = []

    doc = fitz.open(pdf_path)  # Reopen for image extraction

    for element in tqdm(elements, desc="Processing elements"):
        element_type = type(element)

        # Skip unwanted elements
        if element_type in [Footer, Header, Formula]:
            continue

        # Handle images with Azure OpenAI Vision
        if element_type == Image:
            try:
                # Get page and extract image
                page = doc[element.metadata.page_number - 1] if hasattr(element.metadata, 'page_number') else None
                if page:
                    image_list = page.get_images()
                    if image_list:
                        # Get the first image (simplified - you may need better logic)
                        xref = image_list[0][0]
                        base_image = doc.extract_image(xref)
                        image_bytes = base_image["image"]

                        # Extract text using Azure OpenAI Vision
                        extracted_text = extract_image_with_gpt4_vision(image_bytes)
                        processed_elements.append({
                            'type': 'image_text',
                            'content': extracted_text,
                            'page': element.metadata.page_number if hasattr(element.metadata, 'page_number') else None
                        })
            except Exception as e:
                print(f"Error processing image: {e}")
                continue

        # Handle tables with Azure OpenAI Vision
        elif element_type == Table:
            try:
                # Similar to images, extract table region as image and process
                page = doc[element.metadata.page_number - 1] if hasattr(element.metadata, 'page_number') else None
                if page:
                    # Render page region as image (simplified)
                    pix = page.get_pixmap()
                    img_bytes = pix.tobytes("png")

                    # Extract structured table using Azure OpenAI Vision
                    extracted_text = extract_table_with_gpt4_vision(img_bytes)
                    processed_elements.append({
                        'type': 'table_text',
                        'content': extracted_text,
                        'page': element.metadata.page_number if hasattr(element.metadata, 'page_number') else None
                    })
            except Exception as e:
                print(f"Error processing table: {e}")
                # Fallback to original text
                if hasattr(element, 'text') and element.text:
                    processed_elements.append({
                        'type': 'table_fallback',
                        'content': element.text,
                        'page': element.metadata.page_number if hasattr(element.metadata, 'page_number') else None
                    })

        # Handle regular text elements
        elif element_type != PageBreak:
            if hasattr(element, 'text') and element.text and element.text.strip():
                processed_elements.append({
                    'type': 'text',
                    'content': element.text,
                    'page': element.metadata.page_number if hasattr(element.metadata, 'page_number') else None
                })

    doc.close()

    print(f"\nProcessed {len(processed_elements)} elements")
    return processed_elements

---

## 5. Document Chunking and RAG System

After preprocessing, we need to:
1. **Split** the extracted content into manageable chunks
2. **Embed** each chunk using a sentence transformer model
3. **Index** the embeddings for efficient similarity search

### Why Chunking?

Large documents cannot be processed as a whole due to:
- LLM context window limitations
- Need for precise, focused retrieval
- Memory constraints when computing embeddings

Our chunking strategy maintains page-level metadata to enable source tracking.

### The RAG System

The `RAGSystem` class encapsulates the retrieval pipeline:
- Uses **sentence-transformers** for embedding generation
- Stores embeddings in **ChromaDB** for efficient vector similarity search
- Supports configurable top-k retrieval

In [ ]:
def chunk_documents(elements: List[Dict[str, Any]], chunk_size: int = CHUNK_SIZE) -> List[Dict[str, Any]]:
    """
    Split processed elements into chunks for RAG retrieval.
    
    This function aggregates consecutive elements until the chunk size limit
    is reached, preserving page metadata for source tracking.
    
    Args:
        elements: List of processed document elements
        chunk_size: Maximum characters per chunk
        
    Returns:
        List of chunks with id, content, and page metadata
    """
    chunks = []
    current_chunk = ""
    current_page = None
    chunk_id = 0

    for element in elements:
        content = element['content']
        page = element.get('page')

        # If adding this content exceeds chunk size, save current chunk and start new one
        if len(current_chunk) + len(content) > chunk_size and current_chunk:
            chunks.append({
                'id': chunk_id,
                'content': current_chunk.strip(),
                'page': current_page
            })
            chunk_id += 1
            current_chunk = content + " "
            current_page = page
        else:
            current_chunk += content + " "
            if current_page is None:
                current_page = page

    # Add last chunk
    if current_chunk.strip():
        chunks.append({
            'id': chunk_id,
            'content': current_chunk.strip(),
            'page': current_page
        })

    print(f"Created {len(chunks)} chunks")
    return chunks

In [ ]:
class RAGSystem:
    """
    RAG system using sentence transformers and ChromaDB.
    
    This class handles:
    - Embedding generation using sentence-transformers
    - Vector storage and indexing with ChromaDB
    - Similarity-based document retrieval
    
    Attributes:
        embedding_model: The sentence transformer model for embeddings
        client: ChromaDB client for vector storage
        collection: The ChromaDB collection storing document embeddings
    """

    def __init__(self, embedding_model_name: str = EMBEDDING_MODEL):
        """Initialize the RAG system with specified embedding model."""
        self.embedding_model = SentenceTransformer(embedding_model_name)
        self.client = chromadb.Client(Settings(anonymized_telemetry=False))
        self.collection = None

    def create_index(self, chunks: List[Dict[str, Any]], collection_name: str = "compliance_docs"):
        """
        Create vector index from document chunks.
        
        Args:
            chunks: List of document chunks with id, content, and metadata
            collection_name: Name for the ChromaDB collection
        """
        print(f"Creating vector index with {len(chunks)} chunks...")

        # Delete existing collection if it exists
        try:
            self.client.delete_collection(name=collection_name)
        except:
            pass

        # Create new collection with cosine similarity
        self.collection = self.client.create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )

        # Extract texts and generate embeddings
        texts = [chunk['content'] for chunk in chunks]
        embeddings = self.embedding_model.encode(texts, show_progress_bar=True)

        # Add to ChromaDB
        self.collection.add(
            embeddings=embeddings.tolist(),
            documents=texts,
            ids=[str(chunk['id']) for chunk in chunks],
            metadatas=[{'page': chunk.get('page', 'unknown')} for chunk in chunks]
        )

        print("Vector index created successfully")

    def retrieve(self, query: str, top_k: int = TOP_K_DOCUMENTS) -> List[Dict[str, Any]]:
        """
        Retrieve most relevant documents for a query.
        
        Args:
            query: The search query (usually a compliance requirement)
            top_k: Number of documents to retrieve
            
        Returns:
            List of retrieved documents with content, page, and distance metadata
        """
        if self.collection is None:
            raise ValueError("Index not created. Call create_index() first.")

        # Generate query embedding
        query_embedding = self.embedding_model.encode([query])

        # Query ChromaDB
        results = self.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=top_k
        )

        # Format results
        retrieved_docs = []
        for i in range(len(results['ids'][0])):
            retrieved_docs.append({
                'id': results['ids'][0][i],
                'content': results['documents'][0][i],
                'page': results['metadatas'][0][i].get('page', 'unknown'),
                'distance': results['distances'][0][i] if 'distances' in results else None
            })

        return retrieved_docs

---

## 6. Decision Tree Loading

With our RAG system in place, we now need to load the decision trees that encode our compliance requirements. Each decision tree is stored as a PlantUML text file in the `DECISION_TREES_DIR` directory.

The `load_all_decision_trees` function:
1. Scans the directory for `.txt` files
2. Parses each file using our `UMLParser`
3. Returns a dictionary mapping tree names to parsed tree objects

In [ ]:
def load_all_decision_trees(trees_dir: Path) -> Dict[str, Any]:
    """
    Load all decision trees from the specified directory.
    
    Args:
        trees_dir: Path to directory containing PlantUML decision tree files
        
    Returns:
        Dictionary mapping tree names (filenames without extension) to parsed tree objects
    """
    parser = UMLParser()
    trees = {}

    tree_files = list(trees_dir.glob("*.txt"))
    print(f"Found {len(tree_files)} decision tree files")

    for tree_file in tree_files:
        try:
            tree = parser.parse_file(str(tree_file))
            tree_name = tree_file.stem
            trees[tree_name] = tree
            print(f"  ✓ Loaded tree: {tree_name}")
        except Exception as e:
            print(f"  ✗ Error loading {tree_file.name}: {e}")

    return trees

---

## 7. LLM-based Requirement Evaluation

This is the core of our compliance checker: using an LLM to evaluate whether the product documentation satisfies each requirement in the decision tree.

### Chain-of-Thought Prompting

We use a **structured chain-of-thought** approach to ensure:
- **Transparency**: The reasoning process is fully visible
- **Accuracy**: Breaking down analysis reduces errors
- **Traceability**: Each conclusion can be traced to specific evidence

### Evaluation Responses

The LLM must respond with exactly one of:
- **YES**: Documentation clearly satisfies the requirement
- **NO**: Documentation explicitly fails the requirement  
- **NOT APPLICABLE**: Requirement doesn't apply to this product
- **INSUFFICIENT INFO**: Documents lack necessary information

### The EvaluationResult Dataclass

We capture the full evaluation including:
- The final response
- Chain-of-thought reasoning
- Confidence score (0-1)
- Retrieved documents used for evaluation

In [ ]:
@dataclass
class EvaluationResult:
    """Result of evaluating a single requirement.
    
    Attributes:
        response: The evaluation decision (YES/NO/NOT APPLICABLE/INSUFFICIENT INFO)
        reasoning: Chain-of-thought explanation
        confidence: Confidence score between 0 and 1
        retrieved_docs: Documents used for the evaluation
    """
    response: str
    reasoning: str
    confidence: float
    retrieved_docs: List[Dict[str, Any]]


def evaluate_requirement(
    requirement: str,
    rag_system: RAGSystem,
    model: str = LLM_MODEL,
    temperature: float = LLM_TEMPERATURE
) -> EvaluationResult:
    """
    Evaluate a single requirement using RAG + LLM.

    This function:
    1. Retrieves relevant documents using the RAG system
    2. Constructs a structured prompt with chain-of-thought instructions
    3. Calls the LLM to evaluate compliance
    4. Parses the structured response

    Args:
        requirement: The requirement question to evaluate
        rag_system: Initialized RAG system with indexed documents
        model: Azure OpenAI model deployment name
        temperature: Temperature for generation (lower = more deterministic)

    Returns:
        EvaluationResult with response, reasoning, confidence, and retrieved docs
    """
    # Retrieve relevant documents
    retrieved_docs = rag_system.retrieve(requirement, top_k=TOP_K_DOCUMENTS)

    # Build context from retrieved documents
    context = "\n\n".join([
        f"[Document {i+1}, Page {doc['page']}]:\n{doc['content']}"
        for i, doc in enumerate(retrieved_docs)
    ])

    # Create prompt with explicit chain-of-thought structure
    prompt = f"""REQUIREMENT TO EVALUATE:
{requirement}

RELEVANT DOCUMENT EXCERPTS:
{context}

INSTRUCTIONS:
You are evaluating compliance. Let's think through this step by step using a structured chain of thought.

Your response MUST follow this exact structure:

CHAIN OF THOUGHT:

Step 1 - Requirement Analysis:
[What EXACTLY does this requirement ask for? Break it down into specific, verifiable criteria.]

Step 2 - Evidence Gathering:
[What information from the documents is relevant? Quote specific passages and cite document numbers, e.g., "Document 2 states: '...'"]

Step 3 - Gap Analysis:
[Compare the requirement criteria (Step 1) against the evidence (Step 2). What matches? What's missing? What contradicts?]

Step 4 - Preliminary Conclusion:
[Based on the analysis, what seems to be the answer?]

Step 5 - Verification:
[Challenge your conclusion. What assumptions did you make? Are there alternative interpretations? Is the evidence sufficient and unambiguous?]

RESPONSE: [Provide EXACTLY ONE of: YES | NO | NOT APPLICABLE | INSUFFICIENT INFO]

CONFIDENCE: [A decimal between 0.0 and 1.0]

DEFINITIONS:
- YES: The documentation clearly and explicitly satisfies the requirement
- NO: The documentation explicitly contradicts or fails the requirement
- NOT APPLICABLE: The requirement does not apply to this product/context
- INSUFFICIENT INFO: The documents lack the information needed to determine compliance
"""

    # Call Azure OpenAI API
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "system", 
                    "content": "You are a meticulous technical compliance expert. Your task is to analyze product documentation against regulatory requirements using structured reasoning. Always break down your analysis into explicit steps, cite specific evidence from documents, challenge your own assumptions, and provide transparent reasoning before reaching conclusions. Accuracy and traceability are paramount."
                },
                {"role": "user", "content": prompt}
            ],
            temperature=temperature,
            max_tokens=2000  # Increased for detailed chain of thought
        )

        # Parse response
        content = response.choices[0].message.content

        # Extract structured information
        lines = content.strip().split('\n')
        response_value = "INSUFFICIENT INFO"
        confidence = 0.5
        reasoning = ""

        # Find CHAIN OF THOUGHT section (flexible matching)
        cot_start = -1
        for i, line in enumerate(lines):
            if "CHAIN OF THOUGHT" in line.upper() or line.startswith("Step 1"):
                cot_start = i
                break
        
        # Find RESPONSE
        response_idx = -1
        for i, line in enumerate(lines):
            if line.startswith("RESPONSE:"):
                response_idx = i
                response_value = line.replace("RESPONSE:", "").strip()
                # Handle pipe separator format: YES | NO | ...
                response_value = response_value.split('|')[0].strip() if '|' in response_value else response_value
                break
        
        # Extract reasoning (from CHAIN OF THOUGHT to before RESPONSE)
        if cot_start >= 0:
            if response_idx > cot_start:
                reasoning_lines = lines[cot_start:response_idx]
            else:
                reasoning_lines = lines[cot_start:]
            reasoning = "\n".join(reasoning_lines)
            reasoning = reasoning.replace("CHAIN OF THOUGHT:", "").strip()
        
        # Find CONFIDENCE
        for line in lines:
            if line.startswith("CONFIDENCE:"):
                try:
                    conf_str = line.replace("CONFIDENCE:", "").strip()
                    confidence = float(conf_str)
                    # Ensure confidence is in valid range
                    confidence = max(0.0, min(1.0, confidence))
                except:
                    confidence = 0.5
                break

        return EvaluationResult(
            response=response_value,
            reasoning=reasoning,
            confidence=confidence,
            retrieved_docs=retrieved_docs
        )

    except Exception as e:
        print(f"Error calling LLM: {e}")
        return EvaluationResult(
            response="INSUFFICIENT INFO",
            reasoning=f"Error during evaluation: {str(e)}",
            confidence=0.0,
            retrieved_docs=retrieved_docs
        )

---

## 8. Decision Tree Navigation

Now we combine everything: the RAG system provides context, the LLM evaluates requirements, and we use these evaluations to navigate through the decision tree.

### Navigation Logic

Starting from the root node, we:
1. Move to the next node following edges from start/action/join nodes
2. For **decision/switch nodes**, evaluate the requirement using the LLM
3. Based on the response:
   - **YES**: Follow the "Yes" branch
   - **NO**: Follow the "No" branch
   - **NOT APPLICABLE/INSUFFICIENT INFO**: Stop navigation with that as the result
4. Continue until reaching a **verdict** or **terminal** node

### The TreeEvaluationResult

We track the complete evaluation including:
- Tree name and final status (COMPLETED or STOPPED)
- Final verdict (PASS/FAIL/NOT APPLICABLE, or the stopping reason)
- Complete path taken through the tree
- Number of nodes evaluated

In [ ]:
@dataclass
class TreeEvaluationResult:
    """Result of evaluating a complete decision tree.
    
    Attributes:
        tree_name: Name of the decision tree
        status: COMPLETED (reached verdict) or STOPPED (early termination)
        final_verdict: PASS/FAIL/NOT APPLICABLE, or stopping reason
        path_taken: List of nodes visited with their evaluations
        num_nodes_evaluated: Count of decision nodes evaluated
    """
    tree_name: str
    status: str
    final_verdict: Optional[str]
    path_taken: List[Dict[str, Any]]
    num_nodes_evaluated: int


def navigate_decision_tree(
    tree: Any,
    tree_name: str,
    rag_system: RAGSystem
) -> TreeEvaluationResult:
    """
    Navigate a decision tree by evaluating requirements at each decision node.

    The navigation follows these rules:
    - Start at root and follow edges from start/action/join nodes
    - At decision/switch nodes, evaluate the requirement using the LLM
    - YES response: follow the "Yes" branch
    - NO response: follow the "No" branch  
    - NOT APPLICABLE/INSUFFICIENT INFO: stop with that as the verdict
    - Stop when reaching a verdict or terminal node

    Args:
        tree: Parsed UMLDecisionTree object
        tree_name: Name identifier for the tree
        rag_system: Initialized RAG system for document retrieval

    Returns:
        TreeEvaluationResult with complete evaluation details
    """
    print(f"\n{'='*80}")
    print(f"Evaluating Decision Tree: {tree_name}")
    print(f"{'='*80}")

    path_taken = []
    current_node = tree.start
    num_nodes_evaluated = 0

    while current_node:
        node_info = {
            'node_id': current_node.id,
            'node_type': current_node.kind,
            'node_text': current_node.text,
            'evaluation': None
        }

        print(f"\n--- Node {current_node.id} ({current_node.kind}) ---")
        print(f"Text: {current_node.text[:200]}..." if len(current_node.text) > 200 else f"Text: {current_node.text}")

        # Check if we've reached a verdict
        if current_node.kind == "verdict":
            verdict = current_node.text
            print(f"\n✓ Reached verdict: {verdict}")
            node_info['verdict'] = verdict
            path_taken.append(node_info)

            return TreeEvaluationResult(
                tree_name=tree_name,
                status="COMPLETED",
                final_verdict=verdict,
                path_taken=path_taken,
                num_nodes_evaluated=num_nodes_evaluated
            )

        # Check if terminal node
        if current_node.kind == "terminal" or not current_node.edges:
            print("\n✗ Reached terminal node (detach)")
            path_taken.append(node_info)

            return TreeEvaluationResult(
                tree_name=tree_name,
                status="STOPPED",
                final_verdict=None,
                path_taken=path_taken,
                num_nodes_evaluated=num_nodes_evaluated
            )

        # For decision/switch nodes, evaluate requirement
        if current_node.kind in ["decision", "switch"]:
            requirement = current_node.text

            print(f"\nEvaluating requirement...")
            evaluation = evaluate_requirement(requirement, rag_system)
            num_nodes_evaluated += 1

            node_info['evaluation'] = {
                'response': evaluation.response,
                'reasoning': evaluation.reasoning,
                'confidence': evaluation.confidence
            }

            print(f"Response: {evaluation.response}")
            print(f"Confidence: {evaluation.confidence:.2f}")
            print(f"Reasoning: {evaluation.reasoning[:200]}..." if len(evaluation.reasoning) > 200 else f"Reasoning: {evaluation.reasoning}")

            # Decide next step based on response
            next_node = None
            
            if evaluation.response == "YES":
                # For YES response, follow the edge labeled "Yes"
                for edge in current_node.edges:
                    if edge.label and "yes" in edge.label.lower():
                        next_node = edge.target
                        break
                
                # If no "Yes" labeled edge, try first unlabeled (for old format compatibility)
                if not next_node:
                    for edge in current_node.edges:
                        if edge.label is None:
                            next_node = edge.target
                            break

                if next_node:
                    print(f"→ Following 'Yes' branch to node {next_node.id}")
                else:
                    print("✗ No 'Yes' branch found, stopping")
                    
            elif evaluation.response == "NO":
                # For NO response, follow edge labeled "No"
                for edge in current_node.edges:
                    if edge.label and "no" in edge.label.lower():
                        next_node = edge.target
                        break
                
                # If no "No" label, take the first unlabeled edge that is NOT a join node
                if not next_node:
                    for edge in current_node.edges:
                        if edge.label is None and edge.target.kind != "join":
                            next_node = edge.target
                            break
                
                if next_node:
                    print(f"→ Following 'No' branch to node {next_node.id}")
                else:
                    print("✗ No 'No' branch found, stopping")
                    
            else:
                # NOT APPLICABLE or INSUFFICIENT INFO - stop this tree
                print(f"\n✗ Stopping tree due to response: {evaluation.response}")
                path_taken.append(node_info)

                return TreeEvaluationResult(
                    tree_name=tree_name,
                    status="STOPPED",
                    final_verdict=evaluation.response,
                    path_taken=path_taken,
                    num_nodes_evaluated=num_nodes_evaluated
                )
            
            # Continue navigation if we found a next node
            if next_node:
                path_taken.append(node_info)
                current_node = next_node
            else:
                print("✗ Could not find appropriate branch, stopping")
                path_taken.append(node_info)
                break

        # For action/join nodes, just follow the edge
        elif current_node.kind in ["action", "join", "start"]:
            if current_node.edges:
                next_node = current_node.edges[0].target
                print(f"→ Following edge to node {next_node.id}")
                path_taken.append(node_info)
                current_node = next_node
            else:
                print("✗ No edges from this node, stopping")
                path_taken.append(node_info)
                break
        else:
            print(f"⚠ Unknown node type: {current_node.kind}, stopping")
            path_taken.append(node_info)
            break

    # If we exit the loop without reaching a verdict
    return TreeEvaluationResult(
        tree_name=tree_name,
        status="STOPPED",
        final_verdict=None,
        path_taken=path_taken,
        num_nodes_evaluated=num_nodes_evaluated
    )

---

## 9. Main Execution Pipeline

Now we bring everything together in the main execution pipeline. The `run_compliance_check` function orchestrates the complete process:

1. **Preprocess PDF**: Extract and process document elements
2. **Create chunks**: Split content for RAG retrieval
3. **Initialize RAG**: Build the vector index
4. **Load decision trees**: Parse all PlantUML files
5. **Evaluate trees**: Navigate each tree with LLM evaluation
6. **Save results**: Export results to JSON and summary files

### Result Saving

We save two output files:
- **compliance_check_results.json**: Full results with all evaluation details
- **compliance_summary.txt**: Human-readable summary of all tree evaluations

In [ ]:
def save_results(results: Dict[str, TreeEvaluationResult], output_dir: Path):
    """
    Save evaluation results to JSON and summary text files.
    
    Args:
        results: Dictionary mapping tree names to evaluation results
        output_dir: Directory for output files
    """
    output_file = output_dir / "compliance_check_results.json"

    # Convert results to serializable format
    serializable_results = {}
    for tree_name, result in results.items():
        serializable_results[tree_name] = {
            'status': result.status,
            'final_verdict': result.final_verdict,
            'num_nodes_evaluated': result.num_nodes_evaluated,
            'path_taken': result.path_taken
        }

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(serializable_results, f, indent=2, ensure_ascii=False)

    print(f"Results saved to: {output_file}")

    # Also create a summary
    summary_file = output_dir / "compliance_summary.txt"
    with open(summary_file, 'w', encoding='utf-8') as f:
        f.write("COMPLIANCE CHECK SUMMARY\n")
        f.write("=" * 80 + "\n\n")

        for tree_name, result in results.items():
            f.write(f"Tree: {tree_name}\n")
            f.write(f"Status: {result.status}\n")
            f.write(f"Final Verdict: {result.final_verdict or 'N/A'}\n")
            f.write(f"Nodes Evaluated: {result.num_nodes_evaluated}\n")
            f.write("-" * 80 + "\n\n")

    print(f"Summary saved to: {summary_file}")

In [ ]:
def run_compliance_check(
    pdf_path: str,
    trees_dir: Path,
    output_dir: Path
) -> Dict[str, TreeEvaluationResult]:
    """
    Run the complete compliance check pipeline.

    Pipeline steps:
    1. Preprocess PDF with Azure OpenAI Vision
    2. Create document chunks
    3. Initialize RAG system and build index
    4. Load all decision trees
    5. Evaluate each tree
    6. Save results

    Args:
        pdf_path: Path to the PDF document to analyze
        trees_dir: Directory containing PlantUML decision tree files
        output_dir: Directory for saving results

    Returns:
        Dictionary mapping tree names to TreeEvaluationResult objects
    """
    print("=" * 80)
    print("Starting Compliance Check Pipeline")
    print("=" * 80 + "\n")

    # Step 1: Preprocess PDF
    print("Step 1: Preprocessing PDF...")
    processed_elements = preprocess_pdf_with_vision(pdf_path)

    # Step 2: Create chunks
    print("\nStep 2: Creating document chunks...")
    chunks = chunk_documents(processed_elements)

    # Step 3: Initialize RAG system
    print("\nStep 3: Initializing RAG system...")
    rag_system = RAGSystem()
    rag_system.create_index(chunks)

    # Step 4: Load decision trees
    print("\nStep 4: Loading decision trees...")
    trees = load_all_decision_trees(trees_dir)

    # Step 5: Evaluate each tree
    print("\nStep 5: Evaluating decision trees...")
    results = {}

    for tree_name, tree in trees.items():
        result = navigate_decision_tree(tree, tree_name, rag_system)
        results[tree_name] = result

    # Step 6: Save results
    print("\nStep 6: Saving results...")
    save_results(results, output_dir)

    return results

---

## 10. Run the Pipeline

Now let's execute the complete compliance check pipeline. This will:
1. Process the PDF document
2. Build the RAG index
3. Evaluate all decision trees
4. Save the results

> **Note**: This step may take several minutes depending on the size of your PDF and the number of decision trees.

In [ ]:
# Execute the compliance check pipeline
results = run_compliance_check(
    pdf_path=PDF_PATH,
    trees_dir=DECISION_TREES_DIR,
    output_dir=OUTPUT_DIR
)

---

## 11. View Results

Let's examine the results of our compliance check. We'll display:
- A summary of all evaluated trees
- Statistics on completed vs. stopped evaluations
- Individual verdicts for each decision tree

In [ ]:
# Display summary of results
print("\n" + "=" * 80)
print("FINAL RESULTS SUMMARY")
print("=" * 80 + "\n")

completed = []
stopped = []

for tree_name, result in results.items():
    print(f"\n{tree_name}:")
    print(f"  Status: {result.status}")
    print(f"  Final Verdict: {result.final_verdict or 'N/A (stopped early)'}")
    print(f"  Nodes Evaluated: {result.num_nodes_evaluated}")

    if result.status == "COMPLETED":
        completed.append(tree_name)
    else:
        stopped.append(tree_name)

print("\n" + "=" * 80)
print(f"\n✓ Completed Trees ({len(completed)}): {', '.join(completed) if completed else 'None'}")
print(f"✗ Stopped Trees ({len(stopped)}): {', '.join(stopped) if stopped else 'None'}")

---

## 12. Detailed Analysis of a Specific Tree

For deeper insights, let's examine the evaluation path of a specific decision tree. This shows:
- Each node visited during navigation
- The evaluation results for decision nodes
- The chain-of-thought reasoning
- The final verdict reached

This detailed view helps understand *why* a particular verdict was reached.

In [ ]:
# Choose a tree to analyze in detail
tree_to_analyze = list(results.keys())[0] if results else None

if tree_to_analyze:
    result = results[tree_to_analyze]

    print(f"\nDetailed Analysis: {tree_to_analyze}")
    print("=" * 80)

    for i, node in enumerate(result.path_taken, 1):
        print(f"\nStep {i}:")
        print(f"  Node ID: {node['node_id']}")
        print(f"  Type: {node['node_type']}")
        text_preview = node['node_text'][:100] + "..." if len(node['node_text']) > 100 else node['node_text']
        print(f"  Text: {text_preview}")

        if node['evaluation']:
            eval_info = node['evaluation']
            print(f"  Response: {eval_info['response']}")
            print(f"  Confidence: {eval_info['confidence']:.2f}")
            reasoning_preview = eval_info['reasoning'][:200] + "..." if len(eval_info['reasoning']) > 200 else eval_info['reasoning']
            print(f"  Reasoning: {reasoning_preview}")

        if 'verdict' in node:
            print(f"  🎯 VERDICT: {node['verdict']}")
else:
    print("No results available for detailed analysis.")

---

## 13. Testing: Verify Navigation Logic

This section provides a quick test to verify the decision tree navigation is working correctly. We can test with a specific tree to ensure the parser and navigation logic are functioning as expected.

> **Note**: This requires the RAG system to be initialized from the previous execution.

In [ ]:
# Quick test: Verify navigation with a specific tree (e.g., AUM-3)
print("Testing navigation fix on AUM-3...")
parser = UMLParser()

# Try to load a test tree file
test_tree_path = "./assets/DecisionTrees/AUM-3.txt"

if os.path.exists(test_tree_path):
    aum3_tree = parser.parse_file(test_tree_path)

    # Use existing RAG system from previous run
    if 'rag_system' in dir():
        test_result = navigate_decision_tree(aum3_tree, "AUM-3-TEST", rag_system)
        print(f"\n{'='*80}")
        print(f"TEST RESULTS:")
        print(f"Status: {test_result.status}")
        print(f"Final Verdict: {test_result.final_verdict}")
        print(f"Nodes Evaluated: {test_result.num_nodes_evaluated}")
        print(f"Path length: {len(test_result.path_taken)}")
        
        # Show the evaluation responses
        print(f"\n--- Evaluation Path ---")
        for node in test_result.path_taken:
            if node.get('evaluation'):
                print(f"Node {node['node_id']}: {node['evaluation']['response']}")
        print(f"{'='*80}")
    else:
        print("RAG system not initialized. Run the full pipeline first.")
else:
    print(f"Test tree file not found: {test_tree_path}")
    print("Skipping navigation test.")

---

## Conclusion

In this notebook, we implemented a complete **Automated Compliance Checking System** that combines:

1. **Document Processing**: Using Unstructured and GPT-4 Vision for comprehensive PDF parsing
2. **RAG Architecture**: Sentence transformers + ChromaDB for efficient document retrieval
3. **Decision Tree Navigation**: A custom PlantUML parser for encoding compliance logic
4. **LLM Evaluation**: Structured chain-of-thought prompting for transparent reasoning

### Key Features

- **Automated**: Minimal human intervention required
- **Transparent**: Full reasoning trace for each decision
- **Flexible**: Easily extensible to new document types and decision trees
- **Scalable**: ChromaDB enables efficient retrieval over large document collections

### Future Improvements

- Support for more document formats (Word, HTML, etc.)
- Interactive UI for real-time compliance checking
- Fine-tuned models for domain-specific compliance evaluation
- Confidence calibration and uncertainty quantification

---

*This notebook was developed as part of the Information Retrieval course project at the University of Pisa.*